In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# ================= 配置 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DATASET_ROOT = "./dataset_tri_RAD_Enhanced" # 对应新的生成路径
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")
RESULTS_DIR = "./results_rad_dropout"

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 64
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 100

os.makedirs(RESULTS_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_model.pth")

# ================= 1. 数据集 (不变) =================
class InMemoryDataset(Dataset):
    def __init__(self, root_dir):
        self.data = []; self.labels = []
        if not os.path.exists(root_dir): return
        print(f"Loading {root_dir}...")
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists(): 
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in d.glob("*.pt")])
        with tqdm(total=len(files), unit="file") as pbar:
            for p, label in files:
                try: self.data.append(torch.load(p)); self.labels.append(label)
                except: pass
                pbar.update(1)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx], 0.0

# ================= 2. 模型 (含 Channel Dropout) =================

class ChannelDropout(nn.Module):
    """
    随机丢弃 R, A, D 通道，强迫模型学习多模态特征
    """
    def __init__(self, drop_prob=0.5):
        super(ChannelDropout, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training: return x
        
        # x shape: (Batch, 3, H, W)
        batch, channel, _, _ = x.shape
        
        # 生成掩码 (1 = 保留, 0 = 丢弃)
        keep_mask = torch.bernoulli(torch.full((batch, channel, 1, 1), 1 - self.drop_prob, device=x.device))
        
        # 保护机制：如果某样本所有通道都被丢弃，强制保留第一个通道
        mask_sum = keep_mask.sum(dim=1, keepdim=True)
        keep_mask = torch.where(mask_sum == 0, torch.ones_like(keep_mask), keep_mask)
        
        # 数值缩放 (Inverted Dropout)，保持期望值不变
        return x * keep_mask / (1 - self.drop_prob)

class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        
        # [核心] 加入 Channel Dropout
        self.channel_drop = ChannelDropout(drop_prob=0.5)
        
        self.features = nn.Sequential(
            # 输入 3 通道
            nn.Conv2d(3, 16, 3, padding=1), 
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            
            nn.Conv2d(16, 32, 3, padding=1), 
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            
            nn.Conv2d(32, 64, 3, padding=1), 
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.channel_drop(x) # 先做 Dropout
        x = self.features(x)
        x = self.classifier(x)
        return x

# ================= 3. 主流程 =================
def plot_cm(y_true, y_pred, acc):
    cm = confusion_matrix(y_true, y_pred)
    existing_indices = sorted(list(set(y_true) | set(y_pred)))
    names = [CLASS_NAMES[i] for i in existing_indices]
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=names, yticklabels=names)
    plt.title(f"Real Data CM (Acc: {acc:.2%})")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "real_cm.png"))
    plt.close()

def main():
    print("=== Training RAD Model with Channel Dropout ===")
    
    train_ds = InMemoryDataset(SIM_TRAIN_DIR)
    val_ds = InMemoryDataset(SIM_VAL_DIR)
    real_ds = InMemoryDataset(REAL_TEST_DIR)
    
    if len(train_ds) == 0: return

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = TinyCNN(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    for epoch in range(NUM_EPOCHS):
        model.train()
        correct, total = 0, 0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            _, preds = out.max(1)
            correct += preds.eq(labels).sum().item(); total += labels.size(0)
        
        train_acc = correct / total
        
        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                _, preds = out.max(1)
                v_correct += preds.eq(labels).sum().item(); v_total += labels.size(0)
        
        val_acc = v_correct / v_total
        print(f"Epoch {epoch+1} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), BEST_MODEL_PATH)

    # Test
    print("\nTesting Real Data...")
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    model.eval()
    if len(real_ds) > 0:
        loader = DataLoader(real_ds, batch_size=BATCH_SIZE)
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels, _ in loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                _, preds = model(imgs).max(1)
                y_true.extend(labels.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
        
        acc = np.mean(np.array(y_true) == np.array(y_pred))
        print(f"Real Data Accuracy: {acc:.4f}")
        plot_cm(y_true, y_pred, acc)

if __name__ == "__main__":
    main()

=== Training RAD Model with Channel Dropout ===
Loading ./dataset_tri_RAD_Enhanced\sim_train...


100%|██████████| 7685/7685 [00:00<00:00, 11521.80file/s]


Loading ./dataset_tri_RAD_Enhanced\sim_val...


100%|██████████| 1915/1915 [00:00<00:00, 11606.15file/s]


Loading ./dataset_tri_RAD_Enhanced\real_test...


100%|██████████| 4500/4500 [00:00<00:00, 10308.41file/s]


Epoch 1 | Train: 0.2101 | Val: 0.3013
Epoch 2 | Train: 0.3459 | Val: 0.5003
Epoch 3 | Train: 0.4453 | Val: 0.6689
Epoch 4 | Train: 0.4898 | Val: 0.7483
Epoch 5 | Train: 0.5335 | Val: 0.7504
Epoch 6 | Train: 0.5672 | Val: 0.7269
Epoch 7 | Train: 0.5689 | Val: 0.7984
Epoch 8 | Train: 0.5919 | Val: 0.7525
Epoch 9 | Train: 0.5896 | Val: 0.7916
Epoch 10 | Train: 0.6081 | Val: 0.8073
Epoch 11 | Train: 0.6302 | Val: 0.7504
Epoch 12 | Train: 0.6331 | Val: 0.7890
Epoch 13 | Train: 0.6263 | Val: 0.8167
Epoch 14 | Train: 0.6461 | Val: 0.8188
Epoch 15 | Train: 0.6608 | Val: 0.7864
Epoch 16 | Train: 0.6681 | Val: 0.8016
Epoch 17 | Train: 0.6662 | Val: 0.8162
Epoch 18 | Train: 0.6718 | Val: 0.7922
Epoch 19 | Train: 0.6674 | Val: 0.8339
Epoch 20 | Train: 0.6818 | Val: 0.8057
Epoch 21 | Train: 0.6526 | Val: 0.8376
Epoch 22 | Train: 0.6686 | Val: 0.8146
Epoch 23 | Train: 0.6895 | Val: 0.8052
Epoch 24 | Train: 0.6960 | Val: 0.8444
Epoch 25 | Train: 0.6789 | Val: 0.8272
Epoch 26 | Train: 0.6985 | Val: 0.